# Protein Secondary Structure Prediction (SST8)

Bu notebook, amino asit dizisinin her kalıntısı için SST8 etiketi üreten ders projesinin veri kontrolü, ön işleme, çoğunluk sınıfı referansı ve BiLSTM değerlendirme akışını gösterir. Girdi `seq`, hedef `sst8` sütunudur; mevcut `sst3` sütunu öğrenme hedefi olarak kullanılmaz.

Bu yayın sürümündeki güncel `outputs/` dosyaları, mevcut checkpoint'in doğrulama ve üç test dosyasında **yeniden değerlendirilmesiyle** üretilmiştir. Bu hazırlıkta yeni eğitim yapılmamıştır. Notebook'un varsayılan `RUN_TRAINING=False` modu da ağırlıkları yükleyip bütün doğrulama/test dizilerinde tahmin hesaplar; yalnızca kayıtlı tabloları okumaz.

Uyumsuz eski üç epokluk log ve tablolar, açıklamalarıyla `archive/legacy_results/` altında tutulur. Bu dosyalar güncel checkpoint'in eğitim geçmişi olarak kullanılmaz. Notebook hücrelerinde kayıtlı yürütme çıktısı bulunmaz.

Bu paylaşım kopyasında veri CSV’leri ve önceki ağırlık dosyası ayrı tutulur. Dosyaları hazırlamak için `docs/DATA_SOURCES.md` belgesine bak. `RUN_TRAINING=True` seçeneği geçici bir klasörde yeni model eğitir.

## 1. İçe aktarımlar ve yollar

Kod `src/`, veri ve değerlendirme çıktıları proje kökündeki göreli yollarda bulunur. Ayarlar önce `outputs/logs/evaluation_summary.json` içindeki checkpoint yapılandırmasından okunur; eski bir klasörle kullanım için `run_summary.json` geri dönüşü korunur.

Buradaki eğitim hiperparametreleri kayıtlı yapılandırmayı anlatır. Örneğin `epochs`, tamamlanan epok sayısını kanıtlamaz. Güncel değerlendirme özeti gerçek eğitim epok sayısını bilinmeyen olarak kaydeder.

In [ ]:
from __future__ import annotations

import json
import sys
import tempfile
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not ((PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT / "training_secondary_structure_train.csv").is_file()):
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "src").is_dir() and (candidate / "training_secondary_structure_train.csv").is_file():
            PROJECT_ROOT = candidate
            break

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not find the project root containing src/ and the training CSV file.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import build_config
from src.data_loading import discover_schema, load_named_splits
from src.data_validation import compute_split_overlaps, validate_and_clean_split
from src.dataset import ProteinSequenceDataset, create_dataloader
from src.evaluate import evaluate_predictions, predict_with_baseline, predict_with_model
from src.models import BiLSTMTagger, MostFrequentClassBaseline
from src.preprocessing import build_preprocessors
from src.train import train_model_with_encoder
from src.utils import set_seed

config = build_config(PROJECT_ROOT)

run_summary_path = config.paths.logs_dir / "evaluation_summary.json"
if not run_summary_path.exists():
    run_summary_path = config.paths.logs_dir / "run_summary.json"
run_summary = {}
if run_summary_path.exists():
    with run_summary_path.open("r", encoding="utf-8") as file:
        run_summary = json.load(file)
    for field_name, value in run_summary.get("training_config", {}).items():
        if field_name == "length_bucket_edges":
            value = tuple(value)
        setattr(config.training, field_name, value)

set_seed(config.training.seed)
PROJECT_ROOT

## 2. Project pipeline source files

The repository is script-based. `run_all.py` connects the workflow end to end, while the modules under `src/` implement the reusable project logic. This notebook imports and calls those same objects instead of reimplementing a separate simplified version.

In [ ]:
pipeline_files = [
    {
        "stage": "full orchestration",
        "file": "run_all.py",
        "key objects": "main, resolve_device",
        "role in project": "Runs loading, validation, preprocessing, baseline, BiLSTM training, evaluation, and output saving.",
    },
    {
        "stage": "configuration",
        "file": "src/config.py",
        "key objects": "build_config, ProjectPaths, TrainingConfig",
        "role in project": "Centralizes dataset paths, output paths, and training hyperparameters.",
    },
    {
        "stage": "data loading and schema",
        "file": "src/data_loading.py",
        "key objects": "load_named_splits, discover_schema",
        "role in project": "Loads all CSV splits and discovers the sequence, SST8, and SST3 columns.",
    },
    {
        "stage": "data validation",
        "file": "src/data_validation.py",
        "key objects": "validate_and_clean_split, compute_split_overlaps",
        "role in project": "Checks row quality, length alignment, labels, duplicates, distributions, and split overlaps.",
    },
    {
        "stage": "preprocessing",
        "file": "src/preprocessing.py",
        "key objects": "AminoAcidVocab, LabelEncoder, build_preprocessors",
        "role in project": "Builds amino-acid token ids and SST8 label ids from the training split.",
    },
    {
        "stage": "dataset and batching",
        "file": "src/dataset.py",
        "key objects": "ProteinSequenceDataset, create_dataloader",
        "role in project": "Encodes rows and pads variable-length batches with masks for real residues.",
    },
    {
        "stage": "models",
        "file": "src/models.py",
        "key objects": "MostFrequentClassBaseline, BiLSTMTagger",
        "role in project": "Defines the majority-label baseline and the Embedding + BiLSTM + classifier model.",
    },
    {
        "stage": "training",
        "file": "src/train.py",
        "key objects": "train_model_with_encoder",
        "role in project": "Trains the BiLSTM using masked loss, gradient clipping, and validation macro-F1 checkpointing.",
    },
    {
        "stage": "evaluation",
        "file": "src/evaluate.py",
        "key objects": "predict_with_model, predict_with_baseline, evaluate_predictions",
        "role in project": "Creates predictions, token metrics, per-class reports, confusion tables, and sample predictions.",
    },
    {
        "stage": "analysis figures",
        "file": "src/analysis.py",
        "key objects": "plot_class_distribution, plot_training_history, plot_confusion_matrix",
        "role in project": "Generates the saved plots used in the report and presentation.",
    },
]

pipeline_table = pd.DataFrame(pipeline_files)
pipeline_table["exists"] = pipeline_table["file"].apply(lambda relative_path: (PROJECT_ROOT / relative_path).exists())
display(pipeline_table)

## 3. Data loading

The project uses the provided train, validation, and benchmark test CSV files directly. Each row contains an amino-acid sequence and aligned secondary-structure labels.

In [ ]:
raw_splits = load_named_splits(config.paths.split_files)

split_overview = pd.DataFrame(
    [
        {
            "split": split_name,
            "file": path.name,
            "rows": len(raw_splits[split_name]),
            "columns": list(raw_splits[split_name].columns),
        }
        for split_name, path in config.paths.split_files.items()
    ]
)

display(split_overview)
display(raw_splits["train"].head(3))

## 4. Schema discovery and dataset structure

The loader discovers the sequence column and the target column from the CSV schema. The project intentionally trains and evaluates on SST8, while SST3 is kept only as an auxiliary discovered column.

In [ ]:
schema = discover_schema(raw_splits["train"])
config.sequence_column = schema["sequence_column"]
config.target_column = schema["target_column"]
config.auxiliary_target_column = schema.get("auxiliary_target_column", "")

schema_table = pd.DataFrame(
    [
        {"role": "input sequence", "column": config.sequence_column},
        {"role": "main target", "column": config.target_column},
        {"role": "auxiliary target", "column": config.auxiliary_target_column},
    ]
)
columns_by_split = pd.DataFrame(
    {"split": split_name, "columns": ", ".join(dataframe.columns)}
    for split_name, dataframe in raw_splits.items()
)

display(schema_table)
display(columns_by_split)

## 5. Data validation and quality checks

The validation step checks missing values, empty fields, sequence-label length agreement, invalid labels, duplicates, class distributions, and exact sequence overlaps across splits.

In [ ]:
sequence_col = config.sequence_column
target_col = config.target_column

cleaned_splits = {}
validation_summaries = {}

train_cleaned, train_summary = validate_and_clean_split(
    dataframe=raw_splits["train"],
    split_name="train",
    file_path=config.paths.train_csv,
    sequence_column=sequence_col,
    target_column=target_col,
    allowed_label_characters=None,
)
cleaned_splits["train"] = train_cleaned
validation_summaries["train"] = train_summary.to_dict()

allowed_labels = set(train_summary.unique_target_labels)
for split_name in ("validation", "cb513", "ts115", "casp12"):
    cleaned_dataframe, split_summary = validate_and_clean_split(
        dataframe=raw_splits[split_name],
        split_name=split_name,
        file_path=config.paths.split_files[split_name],
        sequence_column=sequence_col,
        target_column=target_col,
        allowed_label_characters=allowed_labels,
    )
    cleaned_splits[split_name] = cleaned_dataframe
    validation_summaries[split_name] = split_summary.to_dict()

validation_table = pd.DataFrame(
    [
        {
            "split": split_name,
            "total_rows": summary["total_rows"],
            "kept_rows": summary["kept_rows"],
            "removed_rows": summary["removed_rows"],
            "length_mismatches": summary["length_mismatches"],
            "duplicate_rows": summary["duplicate_rows"],
            "num_tokens": sum(summary["label_distribution"].values()),
            "min_length": summary["sequence_length_stats"]["min"],
            "max_length": summary["sequence_length_stats"]["max"],
            "mean_length": summary["sequence_length_stats"]["mean"],
        }
        for split_name, summary in validation_summaries.items()
    ]
)
overlap_table = pd.DataFrame(compute_split_overlaps(cleaned_splits, sequence_col))

display(validation_table)
display(overlap_table[overlap_table["overlap_count"] > 0])

In [ ]:
label_distribution = pd.DataFrame(
    {split_name: summary["label_distribution"] for split_name, summary in validation_summaries.items()}
).T.fillna(0).astype(int)

label_percentages = label_distribution.div(label_distribution.sum(axis=1), axis=0).mul(100).round(2)

display(label_distribution)
display(label_percentages)

## 6. Preprocessing

Preprocessing builds an amino-acid vocabulary from the training split and a label encoder for SST8. Padding uses `<PAD>`, unknown symbols use `<UNK>`, and padded labels use `ignore_index=-100` so they do not affect the loss.

In [ ]:
vocab, label_encoder = build_preprocessors(
    train_dataframe=cleaned_splits["train"],
    sequence_column=sequence_col,
    target_column=target_col,
    ignore_index=config.training.ignore_index,
)

preprocessing_summary = pd.DataFrame(
    [
        {"item": "amino_acid_vocab_size", "value": len(vocab.idx_to_token)},
        {"item": "sst8_label_count", "value": len(label_encoder.labels)},
        {"item": "pad_idx", "value": vocab.pad_idx},
        {"item": "unk_idx", "value": vocab.unk_idx},
        {"item": "ignore_index", "value": label_encoder.ignore_index},
    ]
)
vocab_table = pd.DataFrame({"token": vocab.idx_to_token, "id": range(len(vocab.idx_to_token))})
label_table = pd.DataFrame({"label": label_encoder.labels, "id": range(len(label_encoder.labels))})

display(preprocessing_summary)
display(vocab_table)
display(label_table)

In [ ]:
train_dataset = ProteinSequenceDataset(
    dataframe=cleaned_splits["train"],
    sequence_column=sequence_col,
    target_column=target_col,
    vocab=vocab,
    label_encoder=label_encoder,
)
validation_dataset = ProteinSequenceDataset(
    dataframe=cleaned_splits["validation"],
    sequence_column=sequence_col,
    target_column=target_col,
    vocab=vocab,
    label_encoder=label_encoder,
)

preview_loader = create_dataloader(
    dataset=train_dataset,
    batch_size=4,
    shuffle=False,
    pad_idx=vocab.pad_idx,
    ignore_index=config.training.ignore_index,
    num_workers=config.training.num_workers,
)
preview_batch = next(iter(preview_loader))

batch_summary = pd.DataFrame(
    [
        {
            "batch_size": preview_batch["input_ids"].shape[0],
            "padded_length": preview_batch["input_ids"].shape[1],
            "real_tokens": int(preview_batch["mask"].sum().item()),
            "input_tensor_shape": tuple(preview_batch["input_ids"].shape),
            "label_tensor_shape": tuple(preview_batch["labels"].shape),
        }
    ]
)

display(batch_summary)
display(pd.DataFrame({"record_id": preview_batch["record_ids"], "length": preview_batch["lengths"].tolist()}))

## 7. Baseline model

The baseline learns the most frequent SST8 label from the training tokens and predicts that same label at every residue. This gives a simple lower bound for the BiLSTM.

In [ ]:
baseline = MostFrequentClassBaseline()
baseline.fit(cleaned_splits["train"][target_col].tolist())

baseline_rows = []
baseline_bundles = {}
for split_name in ("validation", "cb513", "ts115", "casp12"):
    prediction_records, all_true, all_predicted = predict_with_baseline(
        baseline=baseline,
        dataframe=cleaned_splits[split_name],
        sequence_column=sequence_col,
        target_column=target_col,
    )
    bundle = evaluate_predictions(
        prediction_records=prediction_records,
        all_true=all_true,
        all_predicted=all_predicted,
        label_names=label_encoder.labels,
        split_name=split_name,
        model_name="baseline",
        bucket_edges=config.training.length_bucket_edges,
        sample_count=config.training.sample_prediction_count,
    )
    baseline_bundles[split_name] = bundle
    baseline_rows.append(
        {
            "model": "baseline",
            "split": split_name,
            "accuracy": bundle.metrics["accuracy"],
            "macro_f1": bundle.metrics["macro_f1"],
            "weighted_f1": bundle.metrics["weighted_f1"],
        }
    )

baseline_metrics = pd.DataFrame(baseline_rows)
display(pd.DataFrame({"majority_label": [baseline.majority_label]}))
display(baseline_metrics.round(4))

## 8. BiLSTM model

The main model is an embedding layer followed by a bidirectional LSTM and a linear token classifier. Variable-length protein sequences are handled with padded batches and packed sequence processing.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMTagger(
    vocab_size=len(vocab.idx_to_token),
    embedding_dim=config.training.embedding_dim,
    hidden_dim=config.training.hidden_dim,
    num_labels=len(label_encoder.labels),
    pad_idx=vocab.pad_idx,
    num_layers=config.training.num_layers,
    dropout=config.training.dropout,
).to(device)

model_summary = pd.DataFrame(
    [
        {"item": "device", "value": str(device)},
        {"item": "embedding_dim", "value": config.training.embedding_dim},
        {"item": "hidden_dim", "value": config.training.hidden_dim},
        {"item": "num_layers", "value": config.training.num_layers},
        {"item": "dropout", "value": config.training.dropout},
        {"item": "num_labels", "value": len(label_encoder.labels)},
        {"item": "parameters", "value": sum(parameter.numel() for parameter in model.parameters())},
    ]
)

display(model_summary)
model

## 9. Eğitim mekanizması ve değerlendirme modu

Eğitim fonksiyonu Adam, padding'i dışlayan cross-entropy, gradyan sınırlama, doğrulama macro F1 ile checkpoint seçimi ve erken durdurma uygular. Aşağıdaki tablo bu mekanizmayı ve yapılandırılan sınırları açıklar; güncel ağırlıklar için doğrulanmış bir eğitim eğrisi göstermez.

`RUN_TRAINING=False` varsayılanında yeni eğitim yapılmaz ve eğitim geçmişi CSV'si aranmaz. Güncel `outputs/` altında bu checkpoint ile uyumlu bir eğitim geçmişi bulunmaz. Eski üç epokluk kayıtlar ayrı legacy arşivindedir.

İsteğe bağlı olarak `RUN_TRAINING=True` seçilirse geçici dizine yeni bir model eğitilir ve sonraki bölüm bu yeni ağırlıkları bellekte değerlendirir. Bu işlem yerel checkpoint'i ve sonuç dosyalarını değiştirmez; 11. bölümde gösterilen kayıtlı tablolar/görseller önceki checkpoint'in değerlendirmesine ait olmaya devam eder.

In [ ]:
train_loader = create_dataloader(
    dataset=train_dataset,
    batch_size=config.training.batch_size,
    shuffle=True,
    pad_idx=vocab.pad_idx,
    ignore_index=config.training.ignore_index,
    num_workers=config.training.num_workers,
)
validation_loader = create_dataloader(
    dataset=validation_dataset,
    batch_size=config.training.batch_size,
    shuffle=False,
    pad_idx=vocab.pad_idx,
    ignore_index=config.training.ignore_index,
    num_workers=config.training.num_workers,
)

training_setup = pd.DataFrame(
    [
        {"setting": "configured_max_epochs", "value": config.training.epochs},
        {"setting": "batch_size", "value": config.training.batch_size},
        {"setting": "learning_rate", "value": config.training.learning_rate},
        {"setting": "weight_decay", "value": config.training.weight_decay},
        {"setting": "gradient_clip", "value": config.training.gradient_clip},
        {"setting": "patience", "value": config.training.patience},
        {"setting": "ignore_index", "value": config.training.ignore_index},
    ]
)
display(training_setup)

training_process = pd.DataFrame(
    [
        {"step": 1, "operation": "Create train and validation DataLoaders", "project object": "create_dataloader"},
        {"step": 2, "operation": "Run a forward pass with packed variable-length sequences", "project object": "BiLSTMTagger.forward"},
        {"step": 3, "operation": "Compute cross-entropy while ignoring padded labels", "project object": "torch.nn.CrossEntropyLoss(ignore_index)"},
        {"step": 4, "operation": "Update weights with Adam and gradient clipping", "project object": "train_model_with_encoder"},
        {"step": 5, "operation": "Evaluate on validation after each epoch", "project object": "predict_with_model + evaluate_predictions"},
        {"step": 6, "operation": "Keep the best checkpoint by validation macro F1", "project object": "train_model_with_encoder"},
    ]
)
display(training_process)

RUN_TRAINING = False

if RUN_TRAINING:
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_checkpoint_path = Path(temp_dir) / "bilstm_best.pt"
        history = train_model_with_encoder(
            model=model,
            train_dataloader=train_loader,
            validation_dataloader=validation_loader,
            label_encoder=label_encoder,
            device=device,
            checkpoint_path=temp_checkpoint_path,
            training_config=config.training,
        )
        checkpoint = torch.load(temp_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
else:
    history = None
    display(pd.DataFrame([{
        "mode": "checkpoint re-evaluation",
        "action": "No new training; no compatible training history is available for this checkpoint."
    }]))

if history is not None:
    display(history.round(4))

## 10. Doğrulama ve test dosyalarında değerlendirme

Varsayılan modda yerel checkpoint yüklenir; doğrulama, CB513, TS115 ve CASP12 için tahminler bellekte yeniden hesaplanır. Aynı helper, token accuracy, macro F1, weighted F1, sınıf raporları, karışıklık ve uzunluk grubu tablolarını üretir.

`RUN_TRAINING=True` kullanılmışsa bu bölüm geçici eğitimden gelen yeni ağırlıkları değerlendirir. Bu durumda bellekteki sonuçlar ile 11. bölümdeki yayınlanmış checkpoint çıktıları farklı olabilir.

In [ ]:
checkpoint_path = config.paths.models_dir / "bilstm_best.pt"
if not RUN_TRAINING:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Expected saved checkpoint at {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

loss_function = torch.nn.CrossEntropyLoss(ignore_index=config.training.ignore_index)
evaluation_process = pd.DataFrame(
    [
        {"step": 1, "operation": "Build an encoded dataset and padded DataLoader for each split"},
        {"step": 2, "operation": "Run BiLSTM inference with predict_with_model"},
        {"step": 3, "operation": "Decode predicted label ids back to SST8 strings"},
        {"step": 4, "operation": "Compute token accuracy, macro F1, weighted F1, and loss"},
        {"step": 5, "operation": "Create per-class, confusion, length-bucket, and sample-prediction reports"},
    ]
)
display(evaluation_process)

bilstm_rows = []
bilstm_bundles = {}

for split_name in ("validation", "cb513", "ts115", "casp12"):
    dataset = ProteinSequenceDataset(
        dataframe=cleaned_splits[split_name],
        sequence_column=sequence_col,
        target_column=target_col,
        vocab=vocab,
        label_encoder=label_encoder,
    )
    dataloader = create_dataloader(
        dataset=dataset,
        batch_size=config.training.batch_size,
        shuffle=False,
        pad_idx=vocab.pad_idx,
        ignore_index=config.training.ignore_index,
        num_workers=config.training.num_workers,
    )
    prediction_records, all_true, all_predicted, average_loss = predict_with_model(
        model=model,
        dataloader=dataloader,
        label_encoder=label_encoder,
        device=device,
        loss_function=loss_function,
    )
    bundle = evaluate_predictions(
        prediction_records=prediction_records,
        all_true=all_true,
        all_predicted=all_predicted,
        label_names=label_encoder.labels,
        split_name=split_name,
        model_name="bilstm",
        bucket_edges=config.training.length_bucket_edges,
        sample_count=config.training.sample_prediction_count,
        average_loss=average_loss,
    )
    bilstm_bundles[split_name] = bundle
    bilstm_rows.append(
        {
            "model": "bilstm",
            "split": split_name,
            "accuracy": bundle.metrics["accuracy"],
            "macro_f1": bundle.metrics["macro_f1"],
            "weighted_f1": bundle.metrics["weighted_f1"],
            "loss": bundle.metrics.get("loss"),
        }
    )

bilstm_metrics = pd.DataFrame(bilstm_rows)
comparison = pd.concat([baseline_metrics, bilstm_metrics.drop(columns=["loss"])], ignore_index=True)

display(bilstm_metrics.round(4))
display(comparison.pivot(index="split", columns="model", values=["accuracy", "macro_f1", "weighted_f1"]).round(4))

In [ ]:
validation_bundle = bilstm_bundles["validation"]

display(validation_bundle.per_class_report.round(4))
display(validation_bundle.top_confusions.head(10))
display(validation_bundle.length_bucket_report.round(4))
display(
    validation_bundle.sample_predictions[
        ["record_id", "length", "token_accuracy", "error_count", "mismatch_positions_preview"]
    ]
)

## 11. Kayıtlı güncel değerlendirme çıktıları

Bu bölüm, yayın paketindeki checkpoint'in yeniden değerlendirilmesiyle üretilmiş karşılaştırma tablolarını ve görselleri okur. Güncel çıktılar arasında eğitim eğrisi bulunmaz.

Önceki bölümde isteğe bağlı geçici eğitim seçilmiş olsa bile buradaki dosyalar değişmez. Eski üç epokluk sonuçlar yalnızca `archive/legacy_results/` altında incelenmelidir.

In [ ]:
saved_comparison_path = config.paths.metrics_dir / "model_comparison.csv"
if saved_comparison_path.exists():
    saved_comparison = pd.read_csv(saved_comparison_path)
    display(saved_comparison.round(4))

saved_test_set_path = config.paths.metrics_dir / "test_set_comparison.csv"
if saved_test_set_path.exists():
    saved_test_set_comparison = pd.read_csv(saved_test_set_path)
    display(saved_test_set_comparison.round(4))

figure_paths = [
    config.paths.figures_dir / "class_distribution.png",
    config.paths.figures_dir / "sequence_length_distribution.png",
    config.paths.figures_dir / "confusion_matrix_validation.png",
    config.paths.figures_dir / "test_set_comparison.png",
]

for figure_path in figure_paths:
    if figure_path.exists():
        display(Image(filename=str(figure_path)))

## 12. Değerlendirme

Bu sürüm, mevcut bir checkpoint'in veri ve metriklerle ilişkisini açık biçimde izlemeyi amaçlar. Güncel başarı değerleri yukarıdaki yeniden hesaplanan tablolar ve yayın paketindeki değerlendirme dosyaları üzerinden okunmalıdır. Eski üç epokluk skorlar güncel sonuç olarak kullanılmaz.

Accuracy, macro F1 ve sınıf bazlı recall birlikte değerlendirilmelidir. Çoğunluk sınıfı referansını aşmak, her SST8 sınıfının yeterince öğrenildiğini veya hiperparametrelerin en iyi seçim olduğunu tek başına göstermez. Sınıf dengesizliği ve veri örtüşmeleri yorumların sınırlarıdır.

Checkpoint'in tekrar değerlendirilmesi yeni eğitim yapıldığı, eğitim geçmişinin tamamının doğrulandığı veya nedensel bir mimari karşılaştırma gerçekleştirildiği anlamına gelmez.

## Doğrulama kapsamı

- Veri yükleme, hizalama/tekrar/örtüşme kontrolleri ve eğitimden ön işleme nesneleri kurma adımları notebook'ta yer alır.
- Çoğunluk sınıfı referansı doğrulama ve test dosyalarında hesaplanır.
- Varsayılan mod, yerel checkpoint ile bütün doğrulama ve test dosyalarında tahmin çalıştırır; yeni eğitim yapmaz.
- Eğitim fonksiyonu ve ayarları kaynakta bulunur; bu ağırlıklara ait doğrulanmış bir eğitim eğrisi sunulmaz. İsteğe bağlı geçici eğitim ayrı bir deneydir.
- Güncel kayıtlı metrikler ve şekiller checkpoint'in yeniden değerlendirilmesinden gelir. Uyumsuz eski sonuçlar ayrı legacy arşivinde tutulur.
- Bu notebook'un kayıtlı hücre çıktıları boştur; yayımlanan değerlendirme kanıtları `outputs/` dosyalarında ve deney denetimi belgesinde açıklanır.